# AI BI Copilot — Exploratory Data Analysis

This notebook walks through the three core datasets:
* `transactions` — revenue, cost, and sales data
* `process_events` — order fulfilment event log
* `customer_reviews` — sentiment-enriched reviews

Run `python data/generate_data.py` before executing any cells.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sqlalchemy import create_engine

# Ensure project root is on the path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from modules.db import run_query

sns.set_theme(style='whitegrid')
print('Imports OK')

In [ ]:
# Load transactions
txn = run_query('SELECT * FROM transactions')
print(f'Transactions: {len(txn):,} rows')
txn.head()

In [ ]:
txn.describe()

In [ ]:
# Revenue by region bar chart
rev_region = txn.groupby('region')['revenue'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
rev_region.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Total Revenue by Region', fontsize=14)
ax.set_xlabel('Region')
ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Load process events and calculate cycle times
from modules.process_mining import calculate_cycle_times, load_event_log

raw_df, _ = load_event_log()
print(f'Process events: {len(raw_df):,} rows')
raw_df.head()

In [ ]:
cycle = calculate_cycle_times(raw_df)
print(f'Unique cases: {len(cycle):,}')
cycle['total_cycle_time_minutes'].describe()

In [ ]:
# Load reviews and run sentiment analysis
from modules.sentiment_engine import analyze_reviews_df, get_sentiment_summary

reviews_raw = run_query('SELECT * FROM customer_reviews')
reviews = analyze_reviews_df(reviews_raw)
print(f'Reviews: {len(reviews):,} rows')

summary = get_sentiment_summary(reviews)
print('\nSentiment distribution:')
for label, pct in summary['percentages'].items():
    print(f'  {label}: {pct}%')

In [ ]:
# Sentiment distribution chart
counts = pd.Series(summary['counts'])

fig, ax = plt.subplots(figsize=(6, 6))
colors = {'positive': '#2ecc71', 'neutral': '#f39c12', 'negative': '#e74c3c'}
ax.pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=[colors.get(l, 'grey') for l in counts.index],
    startangle=90,
)
ax.set_title('Customer Review Sentiment Distribution', fontsize=14)
plt.tight_layout()
plt.show()

## Key Insights

* **Revenue concentration**: A small number of regions and product categories drive the majority of revenue.
* **Process bottlenecks**: The Fulfillment and Shipping activities show the highest average durations and occasional extreme outliers that push cases beyond the 2-day SLA.
* **Sentiment skew**: The majority of customer reviews are positive, but negative reviews cluster around specific product categories and deserve targeted attention.

_Replace this placeholder text with your own findings after running the full analysis._